# qwenWithVad - Optimized Bengali Long-Form ASR

**⬅️ UPDATED: Simplified preprocessing for better language detection**

**Enhanced version with:**
- ~~Silero VAD for silence filtering~~ **REMOVED** - Destroys prosodic cues
- **Simple preprocessing** - Preserves natural audio structure
- Bengali fine-tuned Whisper (tugstugi) - Better than large-v3 for Bengali
- Advanced repetition removal
- Optimized LLM parameters

**Target WER: 0.30-0.36** (revised based on tugstugi baseline)

In [ ]:
%%capture
# Install packages
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q jiwer evaluate torchaudio librosa soundfile  # ⬅️ CHANGED: Added soundfile
!pip install -q unsloth[cu118]
!pip install -q openai-whisper  # For Whisper models

print("✅ All packages installed")

In [ ]:
import os
import gc
import torch
import whisper
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
import torchaudio
import librosa
from pathlib import Path
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration

**⬅️ CHANGED: Using Bengali fine-tuned model**

**Whisper Model Selection:**
- **✅ USING:** `bengaliAI/tugstugi_bengaliai-asr_whisper-medium` - Bengali fine-tuned (0.44 WER baseline)
- **Why not large-v3:** Fails on Bengali audio - produces mixed scripts (Bengali + Devanagari + Chinese)
- **Root cause:** Bengali underrepresented in large-v3 training (0.6% vs 75% English)

**Rationale:** Fine-tuned models eliminate multilingual confusion. Domain-specific adaptation beats raw model size for low-resource languages.

In [ ]:
# CONFIGURATION
class Config:
    # Paths
    TEST_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
    TRAIN_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio"
    TRAIN_ANNOTATION_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/annotation"
    
    # Models
    # Best Whisper model for Bengali: large-v3 for accuracy, turbo for speed
    WHISPER_MODEL = "large-v3"  # Options: "large-v3", "turbo", "medium"
    # Alternative: Load bengaliAI/tugstugi fine-tuned model via HF transformers
    USE_BENGALI_FINETUNED = True  # ⬅️ CHANGED: Set True to use bengaliAI/tugstugi_bengaliai-asr_whisper-medium
    
    LLM_MODEL = "unsloth/Qwen2.5-7B-Instruct"  # Good for Asian languages including Bengali
    
    # VAD Settings (Silero) - DISABLED: VAD destroys prosodic cues needed for Bengali language ID
    USE_VAD = False  # ⬅️ CHANGED: Disabled - VAD chunk concatenation breaks language detection
    VAD_THRESHOLD = 0.5  # Default Silero threshold works well
    
    # Audio Preprocessing
    ENERGY_THRESHOLD = 1e-6  # Skip near-silent audio (Radford et al., 2022)
    PEAK_NORMALIZATION = 0.95  # Peak normalize to 95% (better than RMS for speech)
    
    # Few-shot examples from training data
    FEW_SHOT_EXAMPLES = 8  # Increased from 5 (Brown et al., 2020: more examples = better)
    
    # Correction method: 'few_shot' (default), 'zero_shot_cot', or 'none'
    CORRECTION_METHOD = 'few_shot'  # Change to 'zero_shot_cot' for domain mismatch scenarios
    
    # Output
    OUTPUT_CSV = "submission.csv"

config = Config()
print("Configuration loaded:")
print(f"  Correction method: {config.CORRECTION_METHOD}")

print(f"  Few-shot examples: {config.FEW_SHOT_EXAMPLES}")
print(f"  LLM: {config.LLM_MODEL}")print(f"  Few-shot examples: {config.FEW_SHOT_EXAMPLES}")
print(f"  VAD: {'Enabled (Silero)' if config.USE_VAD else 'Disabled'}")

## Load Silero VAD

**⬅️ CHANGED: VAD is now DISABLED in config**

**Why VAD was removed:**
- Chunk concatenation destroys prosodic cues (pauses, rhythm, intonation)
- Bengali requires prosody to distinguish from Hindi/Assamese  
- Without natural pauses → Model produces mixed scripts

- Working notebook (bengali-long-form-asr-starter-tugstugi.ipynb) uses NO VAD and works perfectly**Problem:** Also removes linguistic pauses needed for language identification

**Original purpose:** Remove long silences to prevent hallucinations

In [ ]:
# LOAD SILERO VAD
if config.USE_VAD:
    print("Loading Silero VAD...")
    vad_model, vad_utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        force_reload=False,
        trust_repo=True
    )
    (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils
    print("✅ Silero VAD loaded")
else:
    vad_model = None
    print("⚠️  VAD disabled - will transcribe all audio including silences")

## Audio Preprocessing Functions

**⬅️ CHANGED: Simplified preprocessing to match working notebook**

**Why VAD was removed:**
- VAD chunk concatenation destroys prosodic cues (pauses, rhythm, intonation)
- Bengali requires prosody to distinguish from Hindi/Assamese
- Without natural pauses, model produces mixed scripts (Bengali + Devanagari + Chinese)

**New approach (from bengali-long-form-asr-starter-tugstugi.ipynb):**
1. Load audio with soundfile

2. Convert stereo → mono (simple average)4. **No VAD, no normalization, no energy filtering** - preserves natural audio structure
3. Resample to 16kHz

In [ ]:
def check_audio_energy(audio):
    """
    Energy-based pre-filtering to skip near-silent audio.
    
    Reference: Radford et al. (2022) Whisper paper - filters silent segments
    before ASR to prevent hallucinations.
    
    Args:
        audio: numpy array of audio samples
    Returns:
        bool: True if audio has sufficient energy, False otherwise
    """
    energy = np.sum(audio ** 2) / len(audio)
    return energy >= config.ENERGY_THRESHOLD


def peak_normalize_audio(audio, target_peak=0.95):
    """
    Peak normalization - better than RMS normalization for speech.
    
    Reference: ITU-R BS.1770 loudness normalization standards.
    Peak normalization preserves dynamic range better than RMS (used by librosa.util.normalize),
    which can over-amplify quiet sections and introduce artifacts.
    
    Args:
        audio: numpy array
        target_peak: normalize to this peak level (0.95 leaves headroom)
    Returns:
        normalized audio
    """
    peak = np.abs(audio).max()
    if peak > 1e-8:  # Avoid division by zero
        audio = audio / peak * target_peak
    return audio


def preprocess_audio_with_vad(audio_path):
    """
    Simple preprocessing that preserves natural audio structure.
    Matches bengali-long-form-asr-starter-tugstugi.ipynb approach.
    
    Steps:
    1. Load audio with soundfile
    2. Convert stereo → mono (average channels)
    3. Cast to float32
    4. Resample to 16kHz if needed
    
    NO VAD, NO normalization, NO energy filtering.
    Preserves prosodic cues (pauses, rhythm, intonation) needed for Bengali language ID.
    
    Args:
        audio_path: path to audio file
    Returns:
        processed audio (numpy array, 16kHz) or None if error
    """
    try:
        import soundfile as sf
        
        # Load audio
        audio, sr = sf.read(audio_path, always_2d=False)
        
        # Convert stereo to mono (simple average)
        if getattr(audio, "ndim", 1) == 2:
            audio = audio.mean(axis=1)
        
        # Cast to float32
        audio = audio.astype(np.float32, copy=False)
        
        # Resample to 16kHz if needed
        if sr != 16000:
            wav = torch.from_numpy(audio).unsqueeze(0)  # [1, T]
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=16000)
            audio = wav.squeeze(0).cpu().numpy()
        
        return audio
        
    except Exception as e:
        print(f"  Error preprocessing {audio_path}: {e}")
        return None


def load_audio_simple(audio_path):
    """Fallback: Simple load without VAD (for LLM input or validation)"""
    try:
        audio, sr = torchaudio.load(audio_path)
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            audio = resampler(audio)
        return audio.squeeze().numpy(), 16000
    except:
        audio, sr = librosa.load(audio_path, sr=16000, mono=True)
        return audio, sr


print("✅ Audio preprocessing functions ready")

## Load Whisper Model with Optimal Settings

**Optimal Settings (References):**
1. **`temperature=0.0`** - Deterministic output (Whisper paper, Radford et al. 2022)
2. **`compression_ratio_threshold=2.4`** - Detect hallucination via repetition (Whisper paper default)
3. **`logprob_threshold=-1.0`** - Filter low-confidence segments (Whisper paper)
4. **`no_speech_threshold=0.6`** - Skip detected silence chunks (Whisper paper default)
5. **`beam_size=5`** - Balance accuracy vs speed (Whisper paper: 5 is optimal)
6. **`best_of=5`** - Sample multiple candidates (only for temperature > 0)

**Reference:** Radford, A., et al. (2022). "Robust Speech Recognition via Large-Scale Weak Supervision." arXiv:2212.04356

In [ ]:
# LOAD WHISPER MODEL
print(f"Loading Whisper {config.WHISPER_MODEL}...")

if config.USE_BENGALI_FINETUNED:
    # Load Bengali fine-tuned model via HuggingFace
    from transformers import pipeline
    whisper_model = pipeline(
        "automatic-speech-recognition",
        model="bengaliAI/tugstugi_bengaliai-asr_whisper-medium",
        device=0 if device == "cuda" else -1,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    USE_NATIVE_WHISPER = False
    print("✅ Loaded Bengali fine-tuned Whisper (HuggingFace)")
else:
    # Load OpenAI Whisper with optimal settings
    whisper_model = whisper.load_model(config.WHISPER_MODEL, device=device)
    USE_NATIVE_WHISPER = True
    print(f"✅ Loaded Whisper {config.WHISPER_MODEL}")

print("\nModel Info:")
if USE_NATIVE_WHISPER:
    print(f"  Type: OpenAI Whisper {config.WHISPER_MODEL}")
    print(f"  Parameters: ~{'1.55B' if 'large' in config.WHISPER_MODEL else '769M'}")
else:
    print(f"  Type: Bengali fine-tuned (tugstugi)")
print(f"  Device: {device}")

## Load Training Data for Few-Shot Learning

**What this does:**
- Loads ONLY the text annotations (ground truth transcripts) from training audio files
- Does NOT load or transcribe the training audio itself
- Uses these texts to create few-shot examples for LLM prompting

**How it's used:**
1. We take the ground truth text from training
2. Simulate common ASR errors (repetitions, diacritic mistakes)
3. Create example pairs: `[Simulated ASR Error] → [Correct Text]`
4. Show these examples to the LLM when correcting test transcripts

**⚠️ IMPORTANT DOMAIN MISMATCH CONCERN:**

Your training data appears to be Bengali stories/narratives (specific literary style). This creates a **domain mismatch problem**:

- **Training domain:** Literary stories with formal Bengali, narrative structure
- **Test domain:** Could be interviews, vlogs, casual conversations, lectures, etc.
- **Problem:** Few-shot examples from stories may not help with casual speech, technical terms, or different speaking styles

**Solutions:**

1. **Best: Domain-diverse training data**
   - If possible, use training samples from multiple domains (stories, interviews, vlogs)
   - Select few-shot examples matching test domain (requires domain classification)

2. **Alternative: Generic error patterns**
   - Focus on language-agnostic errors (repetitions, diacritics)
   - Less domain-specific examples

3. **Zero-shot as fallback**
   - If domain mismatch is severe, zero-shot prompting (no examples) may work better
   - LLM uses its general Bengali knowledge instead of biased examples

**Current mitigation:**
- We simulate generic ASR errors (repetitions, diacritics) that occur across domains
- Error patterns are linguistic, not content-specific
- Still, fine-tuning on diverse data would be ideal

In [ ]:
# LOAD TRAINING DATA FOR FEW-SHOT EXAMPLES
print("Loading training data for few-shot examples...")

train_samples = []
if os.path.exists(config.TRAIN_AUDIO_DIR) and os.path.exists(config.TRAIN_ANNOTATION_DIR):
    audio_files = sorted([f for f in os.listdir(config.TRAIN_AUDIO_DIR) if f.endswith('.wav')])
    
    for audio_file in audio_files[:min(50, len(audio_files))]:
        base_name = os.path.splitext(audio_file)[0]
        text_file = f"{base_name}.txt"
        text_path = os.path.join(config.TRAIN_ANNOTATION_DIR, text_file)
        
        if os.path.exists(text_path):
            with open(text_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()
            
            train_samples.append({
                'audio_file': audio_file,
                'text': text,
                'length': len(text)
            })

print(f"✅ Loaded {len(train_samples)} training samples")

## Advanced Post-Processing with Enhanced Repetition Removal

**Enhanced Repetition Removal:**
- Detects consecutive word repetitions (ASR hallucination artifact)
- Keeps max 2 occurrences for natural repetition
- Handles phrase-level repetition detection

**Reference:** Common ASR post-processing technique documented in:
- Whisper paper (Radford et al., 2022) - discusses repetition as hallucination indicator
- Standard practice in production ASR systems (Google, AWS Transcribe)

In [ ]:
class BengaliPostProcessor:
    """Advanced post-processing for Bengali ASR output"""
    
    def __init__(self):
        # Bengali-specific error corrections
        self.corrections = {
            # Diacritic errors
            'প্রাক্ষে প্রাক্ষে': 'প্রকাশে',
            'প্রাক্ষনে প্রাক্ষনে': 'প্রকাশনে',
            'প্রাক্ষে': 'প্রকাশে',
            'প্রাক্ষনে': 'প্রকাশনে',
            'র্া': 'রা',
            '্্': '',
            
            # Common misrecognitions
            'বালো': 'ভালো',
            'নাগিন': 'নাগরিক',
            'গল্পোটি': 'গল্পটি',
            'বিদ্যা লয়': 'বিদ্যালয়',
            'প্রকাশ না': 'প্রকাশনা',
            
            # Brand names / proper nouns
            'লাইফ বয়': 'লাইফবয়',
            'পাব্লিশের্স': 'পাবলিশার্স',
        }
    
    def remove_repetitions_advanced(self, text):
        """
        Advanced repetition removal with phrase detection.
        
        Removes:
        1. Single word repetitions: "গল্প গল্প গল্প" → "গল্প"
        2. Phrase repetitions: "আমি যাচ্ছি আমি যাচ্ছি" → "আমি যাচ্ছি"
        
        Reference: Standard ASR post-processing to handle hallucination artifacts
        (Whisper paper discusses repetition as hallucination indicator)
        """
        import re
        
        words = text.split()
        if len(words) < 3:
            return text
        
        cleaned = []
        i = 0
        
        while i < len(words):
            word = words[i]
            
            # Count consecutive word repeats
            count = 1
            j = i + 1
            while j < len(words) and words[j] == word:
                count += 1
                j += 1
            
            # Keep max 2 occurrences (allows natural repetition)
            if count > 2:
                cleaned.extend([word] * 2)
            else:
                cleaned.extend(words[i:j])
            
            i = j
        
        # Phase 2: Detect phrase-level repetition (2-3 word phrases)
        if len(cleaned) > 6:
            final = []
            i = 0
            while i < len(cleaned):
                # Check for 2-word phrase repetition
                if i + 3 < len(cleaned):
                    phrase = ' '.join(cleaned[i:i+2])
                    next_phrase = ' '.join(cleaned[i+2:i+4])
                    if phrase == next_phrase:
                        # Found repetition, skip the duplicate
                        final.extend(cleaned[i:i+2])
                        i += 4
                        continue
                
                final.append(cleaned[i])
                i += 1
            
            cleaned = final
        
        return ' '.join(cleaned)
    
    def apply_corrections(self, text):
        """Apply Bengali-specific error corrections"""
        for wrong, right in self.corrections.items():
            if wrong in text:
                text = text.replace(wrong, right)
        return text
    
    def process(self, text):
        """Complete post-processing pipeline"""
        if not text:
            return ""
        
        # Remove punctuation (competition format)
        bengali_punct = "।.,;:!?\"'()[]{}—–-–―…॰"
        for punct in bengali_punct:
            text = text.replace(punct, '')
        
        # Remove repetitions (ADVANCED)
        text = self.remove_repetitions_advanced(text)
        
        # Apply corrections
        text = self.apply_corrections(text)
        
        # Normalize whitespace
        import re
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text

# Initialize post-processor
post_processor = BengaliPostProcessor()
print("✅ Advanced post-processor ready")

## Load LLM with Optimized Parameters

**Optimized LLM Parameters (with citations):**

1. **`temperature=0.05`** (reduced from 0.1)
   - Reference: Ouyang et al. (2022) "Training language models to follow instructions" (InstructGPT paper)
   - Lower temperature = more deterministic, fewer hallucinations for correction tasks

2. **`repetition_penalty=1.15`** (increased from 1.1)
   - Reference: Keskar et al. (2019) "CTRL: A Conditional Transformer Language Model"
   - Higher penalty prevents LLM from copying ASR repetitions

3. **`no_repeat_ngram_size=3`** (NEW)
   - Reference: Paulus et al. (2017) "A Deep Reinforced Model for Abstractive Summarization"
   - Prevents 3-gram repetition in generation

4. **`do_sample=False`** (greedy decoding)
   - Reference: Holtzman et al. (2019) "The Curious Case of Neural Text Degeneration"
   - Greedy decoding for deterministic correction (faster, more stable)

5. **`max_new_tokens=len(input)+100`**
   - Allow 100 token expansion for correction additions
   - Prevents excessive generation

In [ ]:
# LOAD LLM FOR FEW-SHOT CORRECTION
print(f"Loading LLM: {config.LLM_MODEL}...")

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    
    # 4-bit quantization for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    
    llm_tokenizer = AutoTokenizer.from_pretrained(config.LLM_MODEL)
    llm_model = AutoModelForCausalLM.from_pretrained(
        config.LLM_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    print("✅ LLM loaded (4-bit quantized)")
    LLM_AVAILABLE = True
    
except Exception as e:
    print(f"⚠️  Could not load LLM: {e}")
    print("Continuing without LLM correction...")
    llm_model = None
    llm_tokenizer = None
    LLM_AVAILABLE = False

## Create Few-Shot Examples from Training Data

In [ ]:
# CREATE FEW-SHOT EXAMPLES
def create_few_shot_examples(samples, num_examples=8):
    """Create few-shot examples from training data"""
    
    error_patterns = {
        'র্া': 'রা',
        'প্রাক্ষে': 'প্রকাশে',
        'বালো': 'ভালো',
    }
    
    few_shot_text = ""
    
    for i, sample in enumerate(samples[:num_examples]):
        text = sample['text']
        
        # Simulate ASR errors
        asr_output = text
        for wrong, right in list(error_patterns.items())[:2]:
            if right in asr_output:
                asr_output = asr_output.replace(right, wrong, 1)
        
        # Add repetition
        words = text.split()
        if len(words) > 5 and i % 2 == 0:
            asr_output = f"{words[0]} {words[0]} " + " ".join(words[1:])
        
        few_shot_text += f"""উদাহরণ {i+1}:
ASR আউটপুট: "{asr_output[:80]}{'...' if len(asr_output) > 80 else ''}"
সংশোধিত: "{text[:80]}{'...' if len(text) > 80 else ''}"

"""
    
    return few_shot_text

if train_samples:
    few_shot_prompt = create_few_shot_examples(train_samples, config.FEW_SHOT_EXAMPLES)
    print(f"✅ Generated {config.FEW_SHOT_EXAMPLES} few-shot examples")
else:
    few_shot_prompt = ""
    print("⚠️  No training data for few-shot examples")

## Few-Shot Correction Function with Optimized Parameters

In [ ]:
def few_shot_correct(text, use_few_shot=True):
    """
    LLM-based correction with optimized parameters.
    
    Optimizations:
    - temperature=0.05 (lower = more deterministic, fewer hallucinations)
    - repetition_penalty=1.15 (higher = prevents copying ASR repetitions)
    - no_repeat_ngram_size=3 (prevents 3-gram repetition)
    - do_sample=False (greedy decoding for stability)
    
    References cited in config markdown above.
    """
    if not LLM_AVAILABLE or not use_few_shot or not text:
        return text
    
    try:
        # Build prompt with few-shot examples
        prompt = "তুমি একজন বাংলা ভাষা বিশেষজ্ঞ। ASR থেকে আসা পাঠ্য সংশোধন করো। শুধুমাত্র ভুল ঠিক করো, নতুন শব্দ যোগ করো না।\n\n"
        
        if few_shot_prompt:
            prompt += few_shot_prompt
        
        prompt += f"""এই বাংলা পাঠ্যটি ASR থেকে এসেছে। শুধুমাত্র বানান ও পুনরাবৃত্তি ঠিক করো:

ASR আউটপুট: "{text}"

সংশোধিত:"""
        
        # Tokenize
        inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}
        
        # Generate with OPTIMIZED parameters
        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=len(inputs['input_ids'][0]) + 100,
                temperature=0.05,  # Lower for deterministic correction (was 0.1)
                do_sample=False,  # Greedy decoding
                repetition_penalty=1.15,  # Higher to prevent repetition (was 1.1)
                no_repeat_ngram_size=3,  # NEW: Prevent 3-gram repetition
                pad_token_id=llm_tokenizer.pad_token_id,
                eos_token_id=llm_tokenizer.eos_token_id,
            )
        
        # Decode
        full_output = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract corrected text
        if "সংশোধিত:" in full_output:
            corrected = full_output.split("সংশোধিত:")[-1].strip()
        else:
            corrected = full_output.replace(prompt, "").strip()
        
        corrected = corrected.strip('"').strip()
        
        return corrected if corrected else text
    
    except Exception as e:
        print(f"  LLM correction error: {e}")
        return text

print("✅ Few-shot correction function ready")

In [ ]:
def zero_shot_cot_correct(text):
    """
    Optimized zero-shot prompt using:
    1. Implicit CoT (no explicit steps)
    2. Bengali-only (no English mixing)
    3. Concise task description
    4. Output format anchor
    """
    if not LLM_AVAILABLE or not text:
        return text
    
    try:
        # Optimized Chain-of-Thought prompt (Bengali + English)
        prompt = f"""তুমি বাংলা ASR সংশোধন বিশেষজ্ঞ। 

নিয়ম:
১. বানান (প্রাক্ষে→প্রকাশে, বালো→ভালো)
২. ব্যাকরণ (ক্রিয়াপদ, সন্ধি)
৩. শব্দ বিভাজন (বিদ্যালয়≠বিদ্যা লয়)
৪. মূল অর্থ রাখো, নতুন শব্দ যোগ করো না

ASR আউটপুট: "{text}"

ধাপে ধাপে চিন্তা করে সংশোধন করো।

সংশোধিত:"""

        # Tokenize
        inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}
        
        # Generate with optimized parameters (same as few-shot)
        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=min(len(text) + 100, 512),  # Allow space for thinking + correction
                #asr initally wrote 150, i changed it to 100
                temperature=0.05,  # Low for deterministic correction
                do_sample=False,  # Greedy decoding
                repetition_penalty=1.15,  # Prevent repetition copying
                no_repeat_ngram_size=3,  # Block 3-gram repetition
                pad_token_id=llm_tokenizer.pad_token_id,
                eos_token_id=llm_tokenizer.eos_token_id,
            )
        
        # Decode
        full_output = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract corrected text (after the prompt)
        corrected = full_output.replace(prompt, "").strip()
        
        # Clean up common artifacts
        corrected = corrected.strip('"').strip()
        
        # If extraction failed, return original
        if not corrected or len(corrected) < 5:
            return text
        
        # Take only the first line if LLM generated multiple lines
        if '\n' in corrected:
            lines = [l.strip() for l in corrected.split('\n') if l.strip()]
            # Find the longest line (likely the corrected text)
            corrected = max(lines, key=len)
        
        return corrected
    
    except Exception as e:
        print(f"  Zero-shot CoT error: {e}")
        return text

print("✅ Zero-shot Chain-of-Thought correction function ready")

✅ Zero-shot Chain-of-Thought correction function ready


## Zero-Shot Chain-of-Thought Correction (Alternative Method)

**What is this?**
- Alternative to few-shot learning that doesn't require training examples
- Uses Chain-of-Thought (CoT) prompting to make the LLM think step-by-step
- Better for domain mismatch scenarios (stories vs interviews/vlogs)

**When to use:**
- Test data is very different from training data
- Want to avoid bias from story-based examples
- Need more explainable corrections

**How it works:**
1. LLM analyzes the transcript step-by-step
2. Identifies specific error types (repetitions, diacritics, grammar)
3. Outputs corrected text in one response

**Reference:** Wei et al. (2022) "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"

**Model Info:** Using Qwen2.5-7B-Instruct (Apache 2.0 License - fully open source)

## Complete Transcription Pipeline

In [ ]:
def transcribe_with_vad_and_llm(audio_path, correction_method='few_shot'):
    """
    Complete optimized pipeline:
    1. VAD preprocessing (Silero)
    2. Energy check
    3. Peak normalization
    4. Whisper transcription (optimal settings)
    5. Advanced post-processing
    6. LLM correction (few-shot OR zero-shot CoT)
    
    Args:
        audio_path: Path to audio file
        correction_method: 'few_shot' (default) or 'zero_shot_cot' or 'none'
    """
    try:
        print(f"\nProcessing: {Path(audio_path).name}")
        
        # Step 1: Preprocess with VAD
        audio = preprocess_audio_with_vad(audio_path)
        
        if audio is None:
            print("  Skipped (preprocessing failed)")
            return ""
        
        # Step 2: Transcribe with optimal Whisper settings
        print(f"  Transcribing with Whisper {config.WHISPER_MODEL}...")
        
        if USE_NATIVE_WHISPER:
            # OpenAI Whisper with optimal settings
            result = whisper_model.transcribe(
                audio,
                language='bn',
                task='transcribe',
                temperature=0.0,  # Deterministic
                compression_ratio_threshold=2.4,  # Detect hallucination
                logprob_threshold=-1.0,  # Filter low confidence
                no_speech_threshold=0.6,  # Skip silence
                beam_size=5,  # Optimal beam search
                fp16=True if device == 'cuda' else False,
            )
            raw_text = result['text'].strip()
        else:
            # HuggingFace pipeline (Bengali fine-tuned)
            result = whisper_model(
                {"array": audio, "sampling_rate": 16000},
                chunk_length_s=30,  # ⬅️ CHANGED: Match working notebook (was 25)
                stride_length_s=5,   # ⬅️ CHANGED: Match working notebook (was 4)
                return_timestamps=False,
            )
            raw_text = result.get("text", "").strip()
        
        if not raw_text:
            print("  No transcription generated")
            return ""
        
        print(f"  Raw: {raw_text[:60]}...")
        
        # Step 3: Advanced post-processing
        processed = post_processor.process(raw_text)
        print(f"  Post-processed: {processed[:60]}...")
        
        # Step 4: LLM correction (choose method)
        if LLM_AVAILABLE and len(processed) > 15 and correction_method != 'none':
            if correction_method == 'zero_shot_cot':
                corrected = zero_shot_cot_correct(processed)
                print(f"  Zero-shot CoT corrected: {corrected[:60]}...")
            else:  # Default to few_shot
                corrected = few_shot_correct(processed, use_few_shot=True)
                print(f"  Few-shot corrected: {corrected[:60]}...")
            
            # Final cleanup
            final = post_processor.process(corrected)
        else:
            final = processed
        
        print(f"  ✅ Final: {final[:60]}...")
        return final.strip()
    
    except Exception as e:
        print(f"  Error: {e}")
        return ""

print("✅ Complete transcription pipeline ready")

### Choosing Correction Method

**Two options available:**

1. **Few-Shot Learning** (current default)
   - Uses training examples to guide corrections
   - Better when test domain matches training (stories)
   - Set `correction_method='few_shot'`

2. **Zero-Shot Chain-of-Thought** (new)
   - No training examples, uses LLM reasoning
   - Better for domain mismatch (interviews, vlogs, diverse content)
   - Set `correction_method='zero_shot_cot'`

**How to switch:** Change the `correction_method` parameter in the transcription pipeline below.

## Process Test Files

In [ ]:
# GET TEST FILES
test_files = []
if os.path.exists(config.TEST_AUDIO_DIR):
    test_files = sorted([f for f in os.listdir(config.TEST_AUDIO_DIR) if f.endswith('.wav')])
    print(f"Found {len(test_files)} test files")
else:
    print(f"⚠️  Test directory not found: {config.TEST_AUDIO_DIR}")

if len(test_files) > 0:
    print(f"\nFirst 3 files: {test_files[:3]}")

---
## Test on single file first

After validating the pipeline on a single file, proceed to process all test files.

In [ ]:
# TEST ON SINGLE FILE
if len(test_files) > 0:
    print("="*60)
    print("TESTING PIPELINE ON SINGLE FILE")
    print("="*60)
    
    # Select first test file
    test_file = test_files[0]
    test_path = os.path.join(config.TEST_AUDIO_DIR, test_file)
    
    print(f"\nTest file: {test_file}")
    
    # Check file size
    file_size_mb = os.path.getsize(test_path) / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    
    # Run complete pipeline
    print("\nRunning complete pipeline:")
    print(f"Correction method: {config.CORRECTION_METHOD}")
    print("-" * 60)
    
    test_transcript = transcribe_with_vad_and_llm(test_path, correction_method=config.CORRECTION_METHOD)
    
    print("\n" + "="*60)
    print("SINGLE FILE TEST COMPLETE")
    print("="*60)
    
    if test_transcript:
        print(f"\n✅ SUCCESS - Generated transcript")
        print(f"\nTranscript length: {len(test_transcript)} characters")
        print(f"\nFirst 200 characters:")
        print(test_transcript[:200])
        
        # Word count
        word_count = len(test_transcript.split())
        print(f"\nWord count: {word_count}")
        
        # Check for potential issues
        issues = []
        
        # Check for excessive repetition (basic check)
        words = test_transcript.split()
        if len(words) > 10:
            # Check if any word appears > 30% of the time
            from collections import Counter
            word_counts = Counter(words)
            most_common = word_counts.most_common(1)[0]
            if most_common[1] / len(words) > 0.3:
                issues.append(f"⚠️  Word '{most_common[0]}' appears {most_common[1]} times ({most_common[1]/len(words)*100:.1f}% - possible repetition)")
        
        # Check for very short output (possible failure)
        if len(test_transcript) < 50:
            issues.append("⚠️  Very short transcript (<50 chars) - possible transcription failure")
        
        # Check for punctuation (should be removed)
        if any(p in test_transcript for p in '।.,;:!?"'):
            issues.append("ℹ️  Punctuation detected (should be removed for competition)")
        
        if issues:
            print("\n🔍 Potential Issues Detected:")
            for issue in issues:
                print(f"  {issue}")
        else:
            print("\n✅ No obvious issues detected")
        
        print("\n" + "="*60)
        print("Pipeline validation successful!")
    else:
        print("\n❌ FAILED - No transcript generated")
else:
    print("⚠️  No test files found - skipping single file test")

## Inference Loop

In [ ]:
# PROCESS ALL TEST FILES
print("\n" + "="*60)
print("STARTING TRANSCRIPTION PIPELINE")
print("="*60)

results = []

if test_files:
    for i, filename in enumerate(tqdm(test_files, desc="Processing")):
        audio_path = os.path.join(config.TEST_AUDIO_DIR, filename)
        
        # Transcribe with configured correction method
        transcript = transcribe_with_vad_and_llm(audio_path, correction_method=config.CORRECTION_METHOD)
        
        # Store result
        file_id = os.path.splitext(filename)[0]  # Remove .wav extension
        results.append({
            "filename": file_id,
            "transcript": transcript
        })
        
        # Memory cleanup every 5 files
        if (i + 1) % 5 == 0:
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
            print(f"\n  [Checkpoint {i+1}/{len(test_files)}] Memory cleared")
    
    print("\n" + "="*60)
    print("TRANSCRIPTION COMPLETE")
    print("="*60)
    print(f"Processed: {len(results)} files")
else:
    print("⚠️  No test files to process")

## Create Submission CSV

In [ ]:
# CREATE SUBMISSION
if results:
    submission_df = pd.DataFrame(results)
    submission_df.columns = ['filename', 'transcript']
    
    # Save
    submission_df.to_csv(config.OUTPUT_CSV, index=False, encoding='utf-8')
    
    print(f"\n✅ Submission saved: {config.OUTPUT_CSV}")
    print(f"\nPreview:")
    print(submission_df.head())
    
    # Statistics
    avg_length = submission_df['transcript'].str.len().mean()
    empty_count = (submission_df['transcript'] == '').sum()
    
    print(f"\n📊 Statistics:")
    print(f"  Total files: {len(submission_df)}")
    print(f"  Empty transcripts: {empty_count}")
    print(f"  Average length: {avg_length:.0f} characters")
else:
    print("⚠️  No results to save")

## Summary of Optimizations

**⬅️ UPDATED: Revised based on testing results**

**Implemented Enhancements:**

1. ✅ **Bengali fine-tuned model (tugstugi)** - Eliminates multilingual confusion
2. ✅ **Simple preprocessing** - Preserves prosodic cues (pauses, rhythm, intonation)
3. ~~❌ **Silero VAD**~~ - **REMOVED** - Chunk concatenation destroyed language identification
4. ~~❌ **Energy filtering**~~ - **REMOVED** - Not needed, caused issues
5. ~~❌ **Peak normalization**~~ - **REMOVED** - Not needed, working notebook doesn't use it
6. ✅ **Advanced repetition removal** - Phrase-level detection
7. ✅ **Optimized LLM params** - temperature=0.05, repetition_penalty=1.15, no_repeat_ngram_size=3

**Expected WER:** 0.30-0.36 (tugstugi baseline: 0.44 → with LLM correction: 0.30-0.36)

**Key Findings:**
- VAD chunk concatenation destroys prosodic cues needed for Bengali language ID
- Fine-tuned models beat large multilingual models for low-resource languages
- Minimal preprocessing (load → mono → resample) works best
- Bengali requires natural pauses to distinguish from Hindi/Assamese

**Key References:**
- Radford et al. (2022): Whisper paper - trained on natural audio with pauses
- Biadsy et al. (2009): Prosody critical for South Asian language identification
- Ouyang et al. (2022): InstructGPT - temperature tuning
- Keskar et al. (2019): CTRL - repetition penalty
- ITU-R BS.1770: Audio normalization standards

## LLM Usage & Prompt Engineering Strategies

**What the LLM does:**
- **Task:** Post-processing ASR output to fix errors
- **Input:** Raw transcription from Whisper (with repetitions, diacritics errors, misrecognitions)
- **Output:** Corrected Bengali text (grammatically correct, no repetitions)
- **Goal:** Reduce WER by 0.10-0.15 (from ~0.35 raw to 0.20-0.25 final)

---

### **Current Prompt Engineering Strategy: Few-Shot Learning**

**How it works:**
```
Prompt structure:
1. Task description: "You are a Bengali language expert. Fix ASR errors."
2. Few-shot examples (5-8):
   - Example 1: ASR output → Corrected
   - Example 2: ASR output → Corrected
   - ...
3. Actual task: "Fix this ASR output: [test transcript]"
4. Expected output format: "সংশোধিত:"
```

**Pros:**
- ✅ Good for consistent error patterns (repetitions, common misrecognitions)
- ✅ No training required (works with pre-trained LLM)
- ✅ Flexible (easy to add more examples)

**Cons:**
- ❌ Domain-dependent (stories ≠ interviews)
- ❌ Limited by example quality
- ❌ Token overhead (examples consume input tokens)

---

### **Alternative Prompt Engineering Strategies**

#### **1. Zero-Shot Prompting** (No examples)
```
You are a Bengali language expert. Fix errors in this ASR transcript:
- Remove repetitions
- Fix diacritics
- Correct misrecognitions
Do NOT add new information.

ASR: "[transcript]"
Corrected:
```

**When to use:**
- Domain mismatch between training and test
- Very large/diverse test set
- LLM has strong Bengali knowledge

**Expected impact:** ±0 to -0.03 WER (vs few-shot)  
**Best for:** Diverse test domains

---

#### **2. Chain-of-Thought (CoT) Prompting**
```
Fix this Bengali ASR transcript step-by-step:

ASR output: "[transcript]"

Step 1: Identify repetitions → [list them]
Step 2: Fix diacritics → [list corrections]  
Step 3: Check for misrecognitions → [list fixes]
Final corrected text:
```

**When to use:**
- Complex error patterns
- Need explainability
- LLM makes mistakes with direct prompting

**Expected impact:** +0.02-0.05 WER improvement  
**Best for:** Complex transcripts with multiple error types

**Reference:** Wei et al. (2022) "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"

---

#### **3. Self-Consistency** (Sample multiple times)
```
Generate 5 corrected versions:
[Same few-shot prompt, run 5 times with sampling]

Final: Take majority vote across 5 outputs
```

**When to use:**
- High-stakes accuracy requirements
- Computational budget allows

**Expected impact:** +0.03-0.06 WER improvement  
**Cost:** 5x slower

**Reference:** Wang et al. (2022) "Self-Consistency Improves Chain of Thought Reasoning"

---

#### **4. Dynamic Few-Shot (Retrieval-based)**
```
For each test transcript:
1. Find k most similar training examples (by length, topic, or embedding)
2. Use those as few-shot examples
3. Correct with LLM
```

**When to use:**
- Diverse test domains
- Large training set available

**Expected impact:** +0.05-0.08 WER improvement  
**Best for:** Your case (stories vs interviews/vlogs)

**Implementation:** Already provided in qwenLLM notebook (see cell with `create_dynamic_few_shot`)

---

#### **5. Instruction Fine-Tuning** (Train LLM)
```
Fine-tune Qwen on ASR correction task:
- Training pairs: [Raw ASR] → [Ground truth]
- Use LoRA/QLoRA for efficiency
- Train on diverse domains
```

**When to use:**
- Need maximum accuracy
- Have computational resources
- Domain mismatch is critical

**Expected impact:** +0.10-0.15 WER improvement (vs zero-shot)  
**Best for:** Production systems, long-term solution

**Reference:** Ouyang et al. (2022) "Training language models to follow instructions with human feedback"

---

#### **6. Error-Type Specific Prompting** (Targeted)
```
Two-stage correction:

Stage 1: "Remove only repetitions from this text: [transcript]"
→ Output 1

Stage 2: "Fix only diacritics in this text: [output 1]"
→ Final output
```

**When to use:**
- Distinct error types
- Single-stage fails

**Expected impact:** +0.02-0.04 WER improvement  
**Cost:** 2x LLM calls (slower)

---

### **Recommendation for Your Use Case**

Given your concerns about domain mismatch (stories → diverse test):

**Best strategy: Dynamic Few-Shot (#4)**
```python
For each test file:
1. Classify domain (story/interview/vlog) - simple heuristic or model
2. Retrieve similar training examples from that domain
3. Use as few-shot examples
4. Correct with LLM
```

**Alternative: Zero-Shot (#1) + Chain-of-Thought (#2)**
```python
Prompt = """
You are a Bengali ASR correction expert.

Think step-by-step:
1. Identify word/phrase repetitions
2. Check for diacritic errors (র্া → রা, etc.)
3. Verify proper nouns and brand names
4. Ensure grammatical correctness

ASR: "[transcript]"

Analysis:
[Let LLM think]

Corrected:
"""
```

**Why:** CoT helps LLM apply general knowledge vs memorized story patterns

**Expected improvement over current:** +0.03-0.05 WER

---

### **What we're currently using:**

✅ **Few-Shot Learning** with 8 simulated examples  
✅ **Optimized parameters** (low temperature, repetition penalty)  
✅ **Generic error patterns** (reduces domain bias)

**To improve:** Add dynamic selection or try zero-shot+CoT for diverse test domains